# Bộ dữ liệu luận văn · lpm-videos

  1 lpm-videos            244 LPM videos (from thivntrung/dataset-video), checked, train/held-out split
  2 slidespeech-devtest   SlideSpeech dev + test videos and the related files (OCR, transcripts)
  3 slidespeech-train100  100 SlideSpeech train videos, streamed out of the 232 GB archive
  4 vn-test12             12 Vietnamese lecture videos from 5 channel types + SponsorBlock labels
  5 m3av-maths            M3AV Oxford Mathematics: slide OCR/LaTeX labels + the 51 videos


In [ ]:
import csv, glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, check=False):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr[-800:])
    return r

def probe(path):
    r = sh(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', str(path)])
    try:
        j = json.loads(r.stdout)
    except Exception:
        return None
    v = [s for s in j.get('streams', []) if s.get('codec_type') == 'video']
    a = [s for s in j.get('streams', []) if s.get('codec_type') == 'audio']
    return {'vcodec': v[0].get('codec_name', '') if v else '', 'width': int(v[0].get('width', 0)) if v else 0,
            'height': int(v[0].get('height', 0)) if v else 0, 'has_audio': bool(a),
            'acodec': a[0].get('codec_name', '') if a else '',
            'duration': round(float(j.get('format', {}).get('duration', 0) or 0), 2)}

def pyav_decodes(path, at=30.0):
    """True if PyAV (what stage 01 uses) can decode a frame of this file."""
    try:
        import av
        with av.open(str(path)) as c:
            s = c.streams.video[0]
            c.seek(int(at / float(s.time_base)), stream=s, backward=True, any_frame=False)
            for frame in c.decode(s):
                frame.to_ndarray(format='rgb24')
                return True
    except Exception as e:
        print('  PyAV lỗi', path.name, str(e)[:120])
    return False

def write_csv(path, rows):
    rows = list(rows)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def du_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 2**30

def install_ytdlp():
    sh([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]'])
    if not shutil.which('deno'):
        os.system('curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1')
        os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']
    print(sh([sys.executable, '-m', 'yt_dlp', '--version']).stdout.strip(), shutil.which('deno'))

BLOCKED_IN_A_ROW = [0]


def youtube(video_id, out, max_height=720, max_sec=None, tries=4, cookies=None, pause=(4, 10)):
    """mp4 with H.264 video (PyAV/OpenCV read it everywhere) and AAC audio; cut to max_sec.
    Uses deno explicitly for YouTube's JS challenge; pauses between videos; after 3 videos in a
    row are refused ("Sign in to confirm you're not a bot") it returns 'blocked' at once."""
    import random
    url = f'https://www.youtube.com/watch?v={video_id}'
    if BLOCKED_IN_A_ROW[0] >= 3:
        return {'status': 'blocked', 'reason': 'dừng: YouTube chặn 3 video liên tiếp'}
    time.sleep(random.uniform(*pause))
    deno = shutil.which('deno') or os.path.expanduser('~/.deno/bin/deno')
    js = ['--js-runtimes', f'deno:{deno}'] if os.path.exists(deno) else []
    formats = [f'bv*[height<={max_height}][vcodec^=avc1]+ba[ext=m4a]',
               f'bv*[height<={max_height}][ext=mp4]+ba[ext=m4a]', f'bv*[height<={max_height}]+ba/b']
    target = Path(out) / f'{video_id}.mp4'
    last = ''
    for attempt in range(tries):
        for f in Path(out).glob(f'{video_id}.*'):
            f.unlink()
        r = sh([sys.executable, '-m', 'yt_dlp', '-q', '--no-warnings', '--ignore-config', *js, '--no-playlist',
                '--socket-timeout', '30', '-f', formats[min(attempt, 2)],
                '--merge-output-format', 'mp4', '--retries', '10', '--fragment-retries', '10',
                *(([('--cookies-from-browser' if cookies[0] == 'browser' else '--cookies'), cookies[1]]) if cookies else []),
                '-o', str(target), url])
        if target.exists():
            info = probe(target)
            if info and info['height'] and info['has_audio']:
                if max_sec and info['duration'] > max_sec + 5:
                    cut = target.with_suffix('.cut.mp4')
                    sh(['ffmpeg', '-v', 'error', '-y', '-i', str(target), '-t', str(max_sec), '-c', 'copy', str(cut)], check=True)
                    original = info['duration']
                    cut.replace(target); info = probe(target); info['original_duration'] = original
                BLOCKED_IN_A_ROW[0] = 0
                return {'status': 'ok', **info}
            last = f'thiếu hình/tiếng: {info}'
        else:
            last = (r.stderr or '').strip().splitlines()[-1:] or ['không có file']
            last = last[0][:200]
            if 'Sign in' in last or 'not a bot' in last:
                BLOCKED_IN_A_ROW[0] += 1
                return {'status': 'blocked', 'reason': last}
        time.sleep(10 * (attempt + 1))
    return {'status': 'failed', 'reason': last}


In [ ]:
SRC = Path(glob.glob('/kaggle/input/**/lecture_dataset/download_log.csv', recursive=True)[0]).parent
SEG = Path(glob.glob('/kaggle/input/**/slides.csv', recursive=True)[0]).parent      # lpm-lecture-slide-segments
OUT = Path('/kaggle/working/lpm-videos'); (OUT / 'video').mkdir(parents=True, exist_ok=True)
print('nguồn video:', SRC, '| mốc slide:', SEG)
sh([sys.executable, '-m', 'pip', 'install', '-q', 'av>=14,<17'])
meta = {r['video_id']: r for r in csv.DictReader(open(SEG / 'videos.csv', encoding='utf-8'))}
# 4 giảng viên để train, 2 giảng viên giữ riêng để test (không dùng để train hay chỉnh ngưỡng)
SPLIT = {'anat-1': 'train', 'bio-1': 'train', 'bio-4': 'train', 'dental': 'train', 'bio-3': 'test', 'anat-2': 'test'}
rows, skipped = [], []
for mp4 in sorted((SRC / 'video').glob('*.mp4')):
    vid, m = mp4.stem, meta.get(mp4.stem)
    info = probe(mp4) or {}
    expected = float(m['duration_s']) if m else 0
    ok = bool(m) and info.get('height') and info.get('has_audio') and abs(info.get('duration', 0) - expected) <= max(10, 0.03 * expected)
    decodes = pyav_decodes(mp4) if ok else False
    if ok and decodes:
        shutil.copy2(mp4, OUT / 'video' / mp4.name)
        rows.append({'video_id': vid, 'speaker': m['speaker'], 'course': m['course'], 'lecture': m['lecture'],
                     'split': SPLIT.get(m['speaker'], 'other'), 'duration_s': info['duration'], 'n_slides': m['n_slides'],
                     'width': info['width'], 'height': info['height'], 'vcodec': info['vcodec'], 'acodec': info['acodec'],
                     'pyav_decodes': decodes, 'file': f'video/{mp4.name}', 'youtube_url': m['youtube_url']})
    else:
        skipped.append({'video_id': vid, 'reason': 'không khớp metadata/thiếu hình tiếng' if not ok else 'PyAV không đọc được'})
write_csv(OUT / 'videos.csv', rows)
have = {r['video_id'] for r in rows}
slides = [r for r in csv.DictReader(open(SEG / 'slides.csv', encoding='utf-8')) if r['video_id'] in have]
for r in slides:
    r['split'] = SPLIT.get(meta[r['video_id']]['speaker'], 'other')
write_csv(OUT / 'slides.csv', slides)
missing = [{**meta[k], 'reason': 'chưa tải được (YouTube yêu cầu xác minh / video bị gỡ)'} for k in meta if k not in have]
write_csv(OUT / 'missing_videos.csv', missing)
if skipped:
    write_csv(OUT / 'skipped.csv', skipped)
from collections import Counter
by = Counter((r['speaker'], r['split']) for r in rows)
summary = {'videos': len(rows), 'hours': round(sum(r['duration_s'] for r in rows) / 3600, 1), 'slides': len(slides),
           'missing': len(missing), 'skipped': len(skipped), 'size_gb': round(du_gb(OUT), 2),
           'vcodec': dict(Counter(r['vcodec'] for r in rows)), 'by_speaker': {f'{a} ({b})': n for (a, b), n in sorted(by.items())},
           'split': dict(Counter(r['split'] for r in rows))}
json.dump(summary, open(OUT / 'summary.json', 'w'), ensure_ascii=False, indent=1)
(OUT / 'README.md').write_text(f"""# LPM lecture videos (cleaned)

Videos of the LPM dataset (Lee et al., ICCV 2023, https://github.com/dondongwon/LPMDataset),
downloaded from YouTube. Each file was checked: video + audio streams, duration matches LPM
(±3 %), and PyAV decodes it. Licence of the annotations: CC BY-NC-SA 4.0; videos stay under
their YouTube licence (research use).

- `video/<id>.mp4` — {summary['videos']} videos, {summary['hours']} h
- `videos.csv` — speaker, course, lecture, **split** (train: anat-1, bio-1, bio-4, dental;
  test: bio-3, anat-2 — held out, never used for training or tuning), duration, codec
- `slides.csv` — {summary['slides']} LPM slide segments (start/end) of these videos, with split
- `missing_videos.csv` — LPM videos not available ({summary['missing']})

Summary: `summary.json`.
""", encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=1))
